In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/loan-prediction-dataset-2025/loan_dataset_20000.csv
/kaggle/input/playground-series-s5e11/sample_submission.csv
/kaggle/input/playground-series-s5e11/train.csv
/kaggle/input/playground-series-s5e11/test.csv
/kaggle/input/pss5e11-model0-oofs/oof_xgb_cv_0.9259837175238498.csv
/kaggle/input/pss5e11-model0-oofs/test_xgb_cv_0.9259837175238498.csv


In [2]:
N_FOLDS = 5
SEED = 42

In [3]:
INPUT_DIR = '/kaggle/input/playground-series-s5e11'
train = pd.read_csv(f'{INPUT_DIR}/train.csv')
train_org = pd.read_csv('/kaggle/input/loan-prediction-dataset-2025/loan_dataset_20000.csv')
test = pd.read_csv(f'{INPUT_DIR}/test.csv')
TARGET = train.columns[-1]
test[TARGET] = -1
# combine

In [4]:
FEATURES = list(train.columns[1:-1])
print(f'FEATURES_{len(FEATURES)}: {FEATURES}, TARGET: {TARGET}')

FEATURES_11: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade'], TARGET: loan_paid_back


In [5]:
train_org = train_org[FEATURES + [TARGET]]
train_org[TARGET] = train_org[TARGET].astype('float64')

# train_org = train_org[FEATURES + [TARGET]].reset_index(drop=True)
# train     = train[FEATURES + [TARGET]].reset_index(drop=True)
# test      = test[FEATURES].reset_index(drop=True)         

# n_org = len(train_org)
# n_tr  = len(train)
# n_te  = len(test)

combine = pd.concat([train_org, train.drop(columns='id'), test.drop(columns='id')], axis=0)

In [6]:
CATS = []
NUMS = []

for c in FEATURES:
    t='CAT'
    if train[c].dtype=='object':
        CATS.append(c)
    else:
        NUMS.append(c)
        t='NUM'

    n = train[c].nunique()
    na = train[c].isna().sum()
    print(f'[{t}] {c} has {n} unique and {na} NA')

print('CATS:', CATS)
print('NUMS:', NUMS)

[NUM] annual_income has 119728 unique and 0 NA
[NUM] debt_to_income_ratio has 526 unique and 0 NA
[NUM] credit_score has 399 unique and 0 NA
[NUM] loan_amount has 111570 unique and 0 NA
[NUM] interest_rate has 1454 unique and 0 NA
[CAT] gender has 3 unique and 0 NA
[CAT] marital_status has 4 unique and 0 NA
[CAT] education_level has 5 unique and 0 NA
[CAT] employment_status has 5 unique and 0 NA
[CAT] loan_purpose has 8 unique and 0 NA
[CAT] grade_subgrade has 30 unique and 0 NA
CATS: ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
NUMS: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']


In [7]:
CATS1 = []
SIZES = {}

for c in CATS:
    # if c in NUMS:
    #     n=f'{c}2'
    #     CATS1.append(n)
    combine[c],_ = combine[c].factorize()
    SIZES[c] = combine[c].max()+1

    # combine[c] = combine[c].astype('int32')
    # combine[n] = combine[n].astype('int32')
for c in NUMS:
    n=f'{c}2'
    combine[n] = combine[c].astype('category')
    CATS1.append(n)
    
print('NEW CATS:', CATS1)
print('CARDINALITY OF ALL CATS:', SIZES)

NEW CATS: ['annual_income2', 'debt_to_income_ratio2', 'credit_score2', 'loan_amount2', 'interest_rate2']
CARDINALITY OF ALL CATS: {'gender': 3, 'marital_status': 4, 'education_level': 5, 'employment_status': 5, 'loan_purpose': 8, 'grade_subgrade': 30}


In [8]:
from itertools import combinations

INTER = []

for col1, col2 in combinations(FEATURES, 2):
    new_col_name = f'{col1}_{col2}'
    INTER.append(new_col_name)
    for df in [combine]:
        df[new_col_name] = df[col1].astype(str) + '_' + df[col2].astype(str)
        
print(f'{len(INTER)} Features.')

55 Features.


In [9]:
# BINS = []
# for col1, col2 in combinations(NUMS, 2):
#     n=f'{col1}_{col2}'
#     for df in [combine]:
#         df[f'{n}_int'] = df[col1] * df[col2]
#         df[f'{n}_add'] = df[col1] + df[col2]
#         df[f'{n}_sub'] = df[col1] - df[col2]
#         df[f'{n}_div'] = df[col1] / (df[col2] + 1e-8)   # add eps to avoid /0
#         df[f'{n}_logdiv'] = np.log1p(df[col1]) - np.log1p(df[col2])
#         df[f'{n}_L2']   = np.sqrt(df[col1]**2 + df[col2]**2)
#         df[f'{n}_L1']   = np.abs(df[col1]) + np.abs(df[col2])
#         df[f'{n}_dist'] = np.abs(df[col1] - df[col2])
#         df[f'{n}_min'] = np.minimum(df[col1], df[col2])
#         df[f'{n}_max'] = np.maximum(df[col1], df[col2])
#         df[f'{n}_avg'] = (df[col1] + df[col2]) / 2
#         df[f'{n}_sq1'] = df[col1]**2
#         df[f'{n}_sq2'] = df[col2]**2
#         df[f'{n}_poly'] = 2 * df[col1] * df[col2]   # 2xy term
#         df[f'{n}_cos'] = np.cos(df[col1] - df[col2])
#         df[f'{n}_sin'] = np.sin(df[col1] - df[col2])
#         df[f'{n}_gt']  = (df[col1] > df[col2]).astype(int)
#         df[f'{n}_eq']  = (np.isclose(df[col1], df[col2])).astype(int)
#         # df[f'{col1}_freq_X_{col2}'] = df[f'{col1}_freq'] * df[col2]



In [10]:
combine['grade_subgrade']

0          0
1          1
2          2
3          3
4          4
          ..
254564    26
254565    11
254566     8
254567    10
254568    12
Name: grade_subgrade, Length: 868563, dtype: int64

In [11]:
def add_domain_features(df):
    """Add professional financial risk assessment features"""
    
    # Monthly income calculation
    df['monthly_income'] = df['annual_income'] / 12
    
    # Estimated monthly payment (using simple interest approximation)
    df['estimated_monthly_payment'] = (df['loan_amount'] * (df['interest_rate'] / 100)) / 12
    
    # Payment to income ratio (key lending metric)
    df['payment_to_income_ratio'] = df['estimated_monthly_payment'] / (df['monthly_income'] + 1)
    
    # Loan to annual income ratio
    df['loan_to_annual_income'] = df['loan_amount'] / (df['annual_income'] + 1)
    
    # High risk flag (industry standard thresholds)
    df['high_risk_flag'] = (
        (df['credit_score'] < 650) & 
        (df['debt_to_income_ratio'] > 0.43)
    ).astype(int)
    
    # Income adequacy
    df['income_adequacy'] = df['annual_income'] / (df['loan_amount'] + 1)
    
    # Estimated total existing debt
    df['estimated_total_debt'] = df['annual_income'] * df['debt_to_income_ratio']
    
    # Remaining income after payment
    df['remaining_income_after_payment'] = df['monthly_income'] - df['estimated_monthly_payment']
    
    return df

def add_synthetic_tells(df):
    """Features that detect synthetic generation artifacts"""
    
    # Perfect correlations detection (synthetic loves clean relationships)
    for col in NUMS:
        # Values that are exact multiples of 100 (too clean)
        df[f'{col}_synthetic_clean'] = (df[col] % 100 == 0).astype(int)
        
        # Values at distribution edges (clipping artifacts)
        q1, q99 = df[col].quantile([0.01, 0.99])
        df[f'{col}_synthetic_clip'] = ((df[col] <= q1) | (df[col] >= q99)).astype(int)
    
    # grade_subgrade encodes generation logic directly
    df['grade'] = df['grade_subgrade'].astype(str).str[0]
    
    # FIXED: Handle single-character grades safely
    grade_str = df['grade_subgrade'].astype(str)
    df['subgrade_num'] = grade_str.str[1:].replace('', '0').astype(int)
    
    # Synthetic data often has perfect conditional independence
    # Capture this with interaction density
    df['synthetic_density'] = df.groupby(CATS)['annual_income'].transform('count')
    
    return df

# combine = add_synthetic_tells(combine)
# CATS1 += ['grade', 'subgrade_num']


combine = add_domain_features(combine)


ROUND = []
rounding_levels = {'1s': 0, '10s': -1}

for col in ['annual_income', 'loan_amount']:
    for suffix, level in rounding_levels.items():
        new_col = f"{col}_ROUND_{suffix}"
        ROUND.append(new_col)
        for df in [combine]:
            df[new_col] = df[col].round(level).astype(int)

print(f"✓ Created {len(ROUND)} rounding features")

✓ Created 4 rounding features


In [12]:
train_org_n = combine.iloc[:len(train_org)]
train_n = combine.iloc[len(train_org):len(train)+len(train_org)]
test_n = combine.iloc[len(train)+len(train_org):]

In [13]:
TE = []
for c in FEATURES:
    tmp = train_org_n.groupby(c)[TARGET].mean()
    tmp_sum = train_org_n.groupby(c)[TARGET].sum()
    tmp_cnt = train_org_n.groupby(c).size()
    
    n = f'TE_{c}'
    n_s = f'TE_sum_{c}'
    n_c = f'TE_count_{c}'
    print(f'{n} , {n_c}', end='')
    tmp.name = n
    tmp_cnt.name = n_c
    tmp_sum.name = n_s

    stats = (pd.concat([tmp,  
                       # tmp_sum,
                        tmp_cnt
                      ]
                      , axis=1,).reset_index().rename(columns={'index': c})) 
    train_org_n = train_org_n.merge(stats, on=c, how='left')
    
    train_n = train_n.merge(stats, on=c, how='left')
    
    test_n = test_n.merge(stats, on=c, how='left')
    
    TE.append(n)
    TE.append(n_c)

TE_annual_income , TE_count_annual_incomeTE_debt_to_income_ratio , TE_count_debt_to_income_ratioTE_credit_score , TE_count_credit_scoreTE_loan_amount , TE_count_loan_amountTE_interest_rate , TE_count_interest_rateTE_gender , TE_count_genderTE_marital_status , TE_count_marital_statusTE_education_level , TE_count_education_levelTE_employment_status , TE_count_employment_statusTE_loan_purpose , TE_count_loan_purposeTE_grade_subgrade , TE_count_grade_subgrade

In [14]:
# BINS = []  # final list of binned column names
# q_list = [5, 10, 15, 20, 25]

# for col in NUMS:  # original numeric columns only
#     for q in q_list:
#         col_name = f"{col}_bin{q}"
        
#         # 1. learn bins on TRAIN_ORG (pure training data)
#         try:
#             _, bins = pd.qcut(
#                 train_org_n[col], q=q, retbins=True, labels=False, duplicates="drop"
#             )
#         except ValueError:  # all values identical
#             bins = np.linspace(train_org_n[col].min(), train_org_n[col].max(), q + 1)

#         # 2. apply identical bins to all three splits
#         train_org_n[col_name] = pd.cut(
#             train_org_n[col], bins=bins, labels=False, include_lowest=True
#         ).astype(np.int8)

#         train_n[col_name] = pd.cut(
#             train_n[col], bins=bins, labels=False, include_lowest=True
#         ).astype(np.int8)

#         test_n[col_name] = pd.cut(
#             test_n[col], bins=bins, labels=False, include_lowest=True
#         ).astype(np.int8)

#         BINS.append(col_name)

# print(f"{len(BINS)} k-bins discretisation features created.")

In [15]:
from sklearn.base import BaseEstimator, TransformerMixin

class TargetEncoder(BaseEstimator, TransformerMixin):
    """
    Target Encoder that supports multiple aggregation functions,
    internal cross-validation for leakage prevention, and smoothing.

    Parameters
    ----------
    cols_to_encode : list of str
        List of column names to be target encoded.

    aggs : list of str, default=['mean']
        List of aggregation functions to apply. Any function accepted by
        pandas' `.agg()` method is supported, such as:
        'mean', 'std', 'var', 'min', 'max', 'skew', 'nunique', 
        'count', 'sum', 'median'.
        Smoothing is applied only to the 'mean' aggregation.

    cv : int, default=5
        Number of folds for cross-validation in fit_transform.

    smooth : float or 'auto', default='auto'
        The smoothing parameter `m`. A larger value puts more weight on the 
        global mean. If 'auto', an empirical Bayes estimate is used.
        
    drop_original : bool, default=False
        If True, the original columns to be encoded are dropped.
    """
    def __init__(self, cols_to_encode, aggs=['mean'], cv=5, smooth='auto', drop_original=False):
        self.cols_to_encode = cols_to_encode
        self.aggs = aggs
        self.cv = cv
        self.smooth = smooth
        self.drop_original = drop_original
        self.mappings_ = {}
        self.global_stats_ = {}

    def fit(self, X, y):
        """
        Learn mappings from the entire dataset.
        These mappings are used for the transform method on validation/test data.
        """
        temp_df = X.copy()
        temp_df['target'] = y

        # Learn global statistics for each aggregation
        for agg_func in self.aggs:
            self.global_stats_[agg_func] = y.agg(agg_func)

        # Learn category-specific mappings
        for col in self.cols_to_encode:
            self.mappings_[col] = {}
            for agg_func in self.aggs:
                mapping = temp_df.groupby(col)['target'].agg(agg_func)
                self.mappings_[col][agg_func] = mapping
        
        return self

    def transform(self, X):
        """
        Apply learned mappings to the data.
        Unseen categories are filled with global statistics.
        """
        X_transformed = X.copy()
        for col in self.cols_to_encode:
            for agg_func in self.aggs:
                new_col_name = f'TE_{col}_{agg_func}'
                map_series = self.mappings_[col][agg_func]
                X_transformed[new_col_name] = X[col].map(map_series)
                X_transformed[new_col_name].fillna(self.global_stats_[agg_func], inplace=True)
        
        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)
            
        return X_transformed

    def fit_transform(self, X, y):
        """
        Fit and transform the data using internal cross-validation to prevent leakage.
        """
        # First, fit on the entire dataset to get global mappings for transform method
        self.fit(X, y)

        # Initialize an empty DataFrame to store encoded features
        encoded_features = pd.DataFrame(index=X.index)
        
        kf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=42)

        for train_idx, val_idx in kf.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val = X.iloc[val_idx]
            
            temp_df_train = X_train.copy()
            temp_df_train['target'] = y_train

            for col in self.cols_to_encode:
                # --- Calculate mappings only on the training part of the fold ---
                for agg_func in self.aggs:
                    new_col_name = f'TE_{col}_{agg_func}'
                    
                    # Calculate global stat for this fold
                    fold_global_stat = y_train.agg(agg_func)
                    
                    # Calculate category stats for this fold
                    mapping = temp_df_train.groupby(col)['target'].agg(agg_func)

                    # --- Apply smoothing only for 'mean' aggregation ---
                    if agg_func == 'mean':
                        counts = temp_df_train.groupby(col)['target'].count()
                        
                        m = self.smooth
                        if self.smooth == 'auto':
                            # Empirical Bayes smoothing
                            variance_between = mapping.var()
                            avg_variance_within = temp_df_train.groupby(col)['target'].var().mean()
                            if variance_between > 0:
                                m = avg_variance_within / variance_between
                            else:
                                m = 0  # No smoothing if no variance between groups
                        
                        # Apply smoothing formula
                        smoothed_mapping = (counts * mapping + m * fold_global_stat) / (counts + m)
                        encoded_values = X_val[col].map(smoothed_mapping)
                    else:
                        encoded_values = X_val[col].map(mapping)
                    
                    # Store encoded values for the validation fold
                    encoded_features.loc[X_val.index, new_col_name] = encoded_values.fillna(fold_global_stat)

        # Merge with original DataFrame
        X_transformed = X.copy()
        for col in encoded_features.columns:
            X_transformed[col] = encoded_features[col]
            
        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)
            
        return X_transformed

In [16]:
# CATS = train.select_dtypes(include='object').columns.to_list()
# train[CATS] = train[CATS].astype('category')
# test[CATS] = test[CATS].astype('category')

# for c in CATS:
#     for df in [train, test]:
#         df[c], _ = df[c].factorize()

In [17]:
X = train_n.drop(columns=[TARGET])
y = train_n[TARGET]
print(X.shape)

(593994, 105)


In [18]:
oofs0 = pd.read_csv('/kaggle/input/pss5e11-model0-oofs/oof_xgb_cv_0.9259837175238498.csv')
test0 = pd.read_csv('/kaggle/input/pss5e11-model0-oofs/test_xgb_cv_0.9259837175238498.csv')

In [19]:
from scipy.special import logit, expit

oofs0_logits = logit(oofs0['loan_paid_back'])
test0_logits = logit(test0['loan_paid_back'])

POSITIVE_LOGIT =  2.0
NEGATIVE_LOGIT = -2.0

true_logits = np.where(y == 1, POSITIVE_LOGIT, NEGATIVE_LOGIT)
residuals = true_logits - oofs0_logits

In [20]:
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 5,
    'colsample_bytree': 0.5,
    'subsample': 0.8,
    'n_estimators': 10000,
    'learning_rate': 0.01,
    'early_stopping_rounds': 100,
    'random_state': 42,
    'n_jobs': -1,
    'device': 'cuda',
    'enable_categorical': True,
}
params_residuals = params.copy()
params_residuals.update({
    'objective':'reg:squarederror',
    'eval_metric': 'rmse',
})
params2 = {     'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 6,
    'min_child_weight': 3,
    'colsample_bytree': 0.3,
    'subsample': 0.6,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'n_estimators': 10000,
    'early_stopping_rounds': 200,
    'random_state': SEED,
    'n_jobs': -1,
    'enable_categorical': True,
    'device': 'cuda',
                      }
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
print(f'X shape: {X.shape}')

# X = X.iloc[:1000]
# y = y.iloc[:1000]

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))


for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f'--- Fold {fold}/{N_FOLDS} ---')
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    residuals_train = residuals.iloc[train_idx]
    residuals_val = residuals.iloc[val_idx]
    # X_test = test[FEATURES].copy()
    X_test = test_n.drop(columns=TARGET).copy()

    TE = TargetEncoder(cols_to_encode=INTER+CATS1, cv=5, smooth='auto', aggs=['mean', 'count'], drop_original=True)
    X_train = TE.fit_transform(X_train, y_train)
    X_val = TE.transform(X_val)
    X_test = TE.transform(X_test)

    TE2 = TargetEncoder(cols_to_encode=ROUND, cv=5, smooth='auto', aggs=['mean', 'count'], drop_original=False)
    X_train = TE2.fit_transform(X_train, y_train)
    X_val = TE2.transform(X_val)
    X_test = TE2.transform(X_test)
    
    # for c in CATS+CATS1:
    #     X_train[c] = X_train[c].astype('category')
    #     X_val[c] = X_val[c].astype('category')
    #     X_test[c] = X_test[c].astype('category')
    print('training shape :', X_train.shape)
    print(f'Residual stats - Mean: {residuals_train.mean():.3f}, Std: {residuals_train.std():.3f}')

    model = XGBClassifier(**params)
    
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              verbose=1000)

    val_preds = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = val_preds
    
    fold_score = roc_auc_score(y_val, val_preds)
    print(f'Fold {fold} AUC: {fold_score:.4f}')
    test_preds += model.predict_proba(X_test)[:,1] / N_FOLDS

# test_preds = expit(test_preds+test0_logits)
overall_auc = roc_auc_score(y, oof_preds)
print(f'====================')
print(f'Overall OOF AUC: {overall_auc:.4f}')
print(f'====================')

X shape: (593994, 105)
--- Fold 1/5 ---
training shape : (475195, 173)
Residual stats - Mean: -0.993, Std: 1.929
[0]	validation_0-auc:0.90885
[1000]	validation_0-auc:0.92697
[1595]	validation_0-auc:0.92721
Fold 1 AUC: 0.9272
--- Fold 2/5 ---
training shape : (475195, 173)
Residual stats - Mean: -1.001, Std: 1.932
[0]	validation_0-auc:0.91527
[1000]	validation_0-auc:0.92791
[1342]	validation_0-auc:0.92798
Fold 2 AUC: 0.9280
--- Fold 3/5 ---
training shape : (475195, 173)
Residual stats - Mean: -0.991, Std: 1.923
[0]	validation_0-auc:0.90871
[1000]	validation_0-auc:0.92574
[1253]	validation_0-auc:0.92579
Fold 3 AUC: 0.9258
--- Fold 4/5 ---
training shape : (475195, 173)
Residual stats - Mean: -0.998, Std: 1.937
[0]	validation_0-auc:0.90811
[1000]	validation_0-auc:0.92684
[1202]	validation_0-auc:0.92687
Fold 4 AUC: 0.9269
--- Fold 5/5 ---
training shape : (475196, 173)
Residual stats - Mean: -1.001, Std: 1.923
[0]	validation_0-auc:0.91473
[1000]	validation_0-auc:0.92636
[1588]	validation_

In [21]:
pd.DataFrame({'id': train.id, TARGET: oof_preds}).to_csv(f'oof_xgb_cv_{overall_auc}.csv', index=False)
pd.DataFrame({'id': test.id, TARGET: test_preds}).to_csv(f'test_xgb_cv_{overall_auc}.csv', index=False)

In [22]:
# import time

# def compute_fast_permutation_importance(model, X_val, y_val, features, n_repeats=2, n_top_features=30):
#     """
#     Fast permutation importance for synthetic data
#     - n_repeats=2 is enough for synthetic data (real patterns are rare)
#     - n_top_features limits computation to most suspicious features
#     """
#     baseline_score = roc_auc_score(y_val, model.predict_proba(X_val)[:,1])
#     importances = {}
    
#     # Focus on high-risk features: TE and interactions
#     if len(features) > n_top_features:
#         features = features[:n_top_features]
    
#     print(f"  Evaluating {len(features)} features...")
    
#     for feature in features:
#         scores = []
#         original_values = X_val[feature].copy()
        
#         for _ in range(n_repeats):
#             # Shuffle feature values
#             X_val[feature] = np.random.permutation(X_val[feature].values)
#             shuffled_score = roc_auc_score(y_val, model.predict_proba(X_val)[:,1])
#             scores.append(baseline_score - shuffled_score)
#             X_val[feature] = original_values
        
#         importances[feature] = np.mean(scores)
    
#     return pd.Series(importances).sort_values(ascending=False)

# # Store original feature lists before any modifications
# INTER_ORIGINAL = INTER.copy()
# CATS1_ORIGINAL = CATS1.copy()
# ROUND_ORIGINAL = ROUND.copy()

# # Modified CV loop with feature importance tracking
# feature_importance_list = []
# oof_preds = np.zeros(len(X))

# print("Phase 1: Training with all features and computing permutation importance...")
# print("=" * 70)

# for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
#     print(f'\n--- Fold {fold}/{N_FOLDS} ---')
    
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#     # Your existing TE pipeline
#     TE = TargetEncoder(cols_to_encode=INTER_ORIGINAL+CATS1_ORIGINAL, cv=5, smooth='auto', aggs=['mean', 'count'], drop_original=True)
#     X_train = TE.fit_transform(X_train, y_train)
#     X_val = TE.transform(X_val)

#     TE2 = TargetEncoder(cols_to_encode=ROUND_ORIGINAL, cv=5, smooth='auto', aggs=['mean', 'count'], drop_original=False)
#     X_train = TE2.fit_transform(X_train, y_train)
#     X_val = TE2.transform(X_val)
    
#     print('Training shape:', X_train.shape)
    
#     # Train model
#     model = XGBClassifier(**params)
#     model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=0)
    
#     val_preds = model.predict_proba(X_val)[:,1]
#     oof_preds[val_idx] = val_preds
    
#     fold_score = roc_auc_score(y_val, val_preds)
#     print(f'Fold {fold} AUC: {fold_score:.4f}')
    
#     # === PERMUTATION IMPORTANCE ===
#     # Only evaluate features that are likely noise: TE and interactions
#     features_to_check = [c for c in X_val.columns if c.startswith('TE_') or '_' in c]
    
#     # Fast evaluation (takes ~30-60 seconds per fold)
#     print(f'\nComputing permutation importance (Fold {fold})...')
#     start = time.time()
#     fold_importance = compute_fast_permutation_importance(
#         model, X_val.copy(), y_val, 
#         features_to_check, 
#         n_repeats=2,
#         n_top_features=30  # Limit to top 30 suspects
#     )
#     print(f'Completed in {time.time()-start:.1f}s')
#     print(f'Top features: {fold_importance.head(5).to_dict()}')
#     print(f'Weakest features: {fold_importance.tail(3).to_dict()}')
    
#     feature_importance_list.append(fold_importance)

# # === POST-CV ANALYSIS ===
# print('\n' + '='*70)
# print('FEATURE IMPORTANCE ANALYSIS')
# print('='*70)

# # Average across folds
# mean_importance = pd.concat(feature_importance_list, axis=1).mean(axis=1).sort_values(ascending=False)

# print(f'\nTop 10 Most Important Features:')
# print(mean_importance.head(10))

# print(f'\nBottom 10 Least Important Features:')
# print(mean_importance.tail(10))

# # **CRITICAL SYNTHETIC DATA RULE**: Drop features with NEGATIVE importance
# # (they hurt performance when shuffled = they were overfitting)
# importance_threshold = max(0.0001, mean_importance.quantile(0.15))  # Drop bottom 15%
# features_to_drop = mean_importance[mean_importance < -0.0001].index.tolist() + \
#                    mean_importance[mean_importance < importance_threshold].index.tolist()

# print(f'\n🔥 Features to DROP ({len(features_to_drop)}): {features_to_drop}')

# # === RETRAIN WITH SELECTED FEATURES ===
# if len(features_to_drop) > 0:
#     print('\n' + '='*70)
#     print('RETRAINING WITH CLEAN FEATURES')
#     print('='*70)
    
#     # CRITICAL: Filter feature lists to remove dropped features
#     # This prevents KeyError during TE
#     INTER_FILTERED = [c for c in INTER_ORIGINAL if c not in features_to_drop]
#     CATS1_FILTERED = [c for c in CATS1_ORIGINAL if c not in features_to_drop]
#     ROUND_FILTERED = [c for c in ROUND_ORIGINAL if c not in features_to_drop]
    
#     # Clean datasets
#     X_clean = X.drop(columns=features_to_drop)
#     test_clean = test_n.drop(columns=TARGET).drop(columns=features_to_drop)
    
#     print(f'Feature count: {X.shape[1]} → {X_clean.shape[1]}')
#     print(f'TE features: {len(INTER_ORIGINAL)+len(CATS1_ORIGINAL)} → {len(INTER_FILTERED)+len(CATS1_FILTERED)}')
#     print(f'Round features: {len(ROUND_ORIGINAL)} → {len(ROUND_FILTERED)}')
    
#     oof_clean = np.zeros(len(X_clean))
#     test_clean_preds = np.zeros(len(test_clean))
    
#     for fold, (train_idx, val_idx) in enumerate(skf.split(X_clean, y), 1):
#         print(f'--- Fold {fold}/{N_FOLDS} ---')
        
#         X_train, X_val = X_clean.iloc[train_idx], X_clean.iloc[val_idx]
#         y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
#         # Use filtered feature lists - this fixes the KeyError
#         if INTER_FILTERED or CATS1_FILTERED:
#             TE = TargetEncoder(
#                 cols_to_encode=INTER_FILTERED + CATS1_FILTERED, 
#                 cv=5, smooth='auto', aggs=['mean'],  # Drop 'count' as it's mostly noise
#                 drop_original=True
#             )
#             X_train = TE.fit_transform(X_train, y_train)
#             X_val = TE.transform(X_val)
#             X_test_transformed = TE.transform(test_clean)
#         else:
#             X_test_transformed = test_clean.copy()

#         if ROUND_FILTERED:
#             TE2 = TargetEncoder(
#                 cols_to_encode=ROUND_FILTERED, 
#                 cv=5, smooth='auto', aggs=['mean'],  # Drop 'count' encoding
#                 drop_original=False
#             )
#             X_train = TE2.fit_transform(X_train, y_train)
#             X_val = TE2.transform(X_val)
#             X_test_transformed = TE2.transform(X_test_transformed)
        
#         model = XGBClassifier(**params)
#         model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=0)
        
#         val_preds = model.predict_proba(X_val)[:,1]
#         oof_clean[val_idx] = val_preds
#         test_clean_preds += model.predict_proba(X_test_transformed)[:,1] / N_FOLDS
    
#     clean_auc = roc_auc_score(y, oof_clean)
#     original_auc = roc_auc_score(y, oof_preds)
#     print(f'\n====================')
#     print(f'Original OOF AUC: {original_auc:.5f}')
#     print(f'Cleaned OOF AUC:  {clean_auc:.5f}')
#     print(f'IMPROVEMENT:      {clean_auc - original_auc:+.5f}')
#     if clean_auc > original_auc:
#         print('✅ Feature cleaning WORKED!')
#     else:
#         print('❌ No improvement - consider dropping fewer features')
#     print(f'====================')
# else:
#     print("\nNo features to drop. All features are clean.")

In [23]:
# # Typical output you'll see:
# print(mean_importance.head(15))
# # Expected survivors:
# # TE_grade_subgrade_mean          0.00234  (original grade signal)
# # interest_rate                   0.00189  (clean numeric)
# # loan_to_annual_income           0.00156  (domain feature)
# # credit_score_debt_to_income_hash 0.00123 (good interaction)

# print(mean_importance.tail(15))
# # Expected drops:
# # TE_annual_income_debt_to_income_interest_rate_mean  -0.00012 (overfit trio)
# # annual_income_ROUND_10s                              0.00004 (useless rounding)
# # TE_marital_status_education_level_count             -0.00008 (noise)

In [24]:
# # =====  LGBoost cell – TE fit inside, no external reuse  =====
# import lightgbm as lgb
# from sklearn.metrics import roc_auc_score

# lgb_params = {
#     'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
#     'num_leaves': 31, 'max_depth': 5, 'learning_rate': 0.01,
#     'n_estimators': 10000, 'subsample': 0.8, 'colsample_bytree': 0.5,
#     'reg_alpha': 0.001, 'reg_lambda': 0.001, 'random_state': 42,
#     'n_jobs': -1, 'verbose': -1, 'early_stopping_rounds': 100
# }

# oof_lgb = np.zeros(len(X))
# test_lgb = np.zeros(len(test))

# for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
#     print(f'--- LGB Fold {fold}/{N_FOLDS} ---')
    
#     X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
#     X_te = test_n.drop(columns=[TARGET])

#     # TargetEncoder fitted **inside** this cell only
#     te = TargetEncoder(cols_to_encode=INTER, cv=5, smooth='auto',
#                        aggs=['mean'], drop_original=True)
#     X_tr = te.fit_transform(X_tr, y_tr)
#     X_val = te.transform(X_val)
#     X_te  = te.transform(X_te)

#     # categorical dtype
#     for col in CATS:
#         X_tr[col] = X_tr[col].astype('category')
#         X_val[col] = X_val[col].astype('category')
#         X_te[col] = X_te[col].astype('category')

#     train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=CATS)
#     val_data   = lgb.Dataset(X_val, label=y_val, reference=train_data)

#     model = lgb.train(lgb_params, train_data, valid_sets=[val_data],
#                       callbacks=[lgb.early_stopping(100), lgb.log_evaluation(1000)])

#     val_pred = model.predict(X_val, num_iteration=model.best_iteration)
#     oof_lgb[val_idx] = val_pred
#     test_lgb += model.predict(X_te, num_iteration=model.best_iteration) / N_FOLDS

#     print(f'LGB Fold {fold} AUC: {roc_auc_score(y_val, val_pred):.4f}')

# overall_lgb = roc_auc_score(y, oof_lgb)
# print(f'====================\nLGB OOF AUC: {overall_lgb:.4f}\n====================')

# # save
# pd.DataFrame({'id': train.id, TARGET: oof_lgb}) \
#     .to_csv(f'oof_lgb_cv_{overall_lgb:.4f}.csv', index=False)
# pd.DataFrame({'id': test.id, TARGET: test_lgb}) \
#     .to_csv(f'test_lgb_cv_{overall_lgb:.4f}.csv', index=False)

In [25]:
# X = X.iloc[:1000]
# y = y.iloc[:1000]

In [26]:
# # =====  SK-MLP  (no cats handling)  =====
# from sklearn.neural_network import MLPClassifier
# from sklearn.preprocessing import RobustScaler
# from sklearn.pipeline import Pipeline

# oof_mlp = np.zeros(len(X))
# test_mlp = np.zeros(len(test))

# X_m = X.fillna(0)
# test_m = test_n.fillna(0)

# for fold, (train_idx, val_idx) in enumerate(skf.split(X_m, y), 1):
#     print(f'--- MLP Fold {fold}/{N_FOLDS} ---')
#     X_tr, X_val = X_m.iloc[train_idx], X_m.iloc[val_idx]
#     y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
#     X_te = test_m.drop(columns=[TARGET])

#     te = TargetEncoder(cols_to_encode=INTER, cv=5, smooth='auto',
#                        aggs=['mean'], drop_original=True)
#     X_tr = te.fit_transform(X_tr, y_tr)
#     X_val = te.transform(X_val)
#     X_te  = te.transform(X_te)

#     model = Pipeline([
#         ('sc', RobustScaler()),
#         ('mlp', MLPClassifier(
#             hidden_layer_sizes=(128, 64, 32), max_iter=1000,
#             learning_rate_init=0.01, early_stopping=True,
#             validation_fraction=0.15, n_iter_no_change=50,
#             random_state=42, verbose=False))
#     ])
#     model.fit(X_tr, y_tr)
#     val_pred = model.predict_proba(X_val)[:, 1]
#     oof_mlp[val_idx] = val_pred
#     test_mlp += model.predict_proba(X_te)[:, 1] / N_FOLDS
#     print(f'MLP Fold {fold} AUC: {roc_auc_score(y_val, val_pred):.4f}')

# overall_mlp = roc_auc_score(y, oof_mlp)
# print(f'====================\nMLP OOF AUC: {overall_mlp:.4f}\n====================')
# pd.DataFrame({'id': train.id, TARGET: oof_mlp}).to_csv(f'oof_mlp_cv_{overall_mlp:.4f}.csv', index=False)
# pd.DataFrame({'id': test.id, TARGET: test_mlp}).to_csv(f'test_mlp_cv_{overall_mlp:.4f}.csv', index=False)

In [27]:
# # =====  CATBOOST  =====
# from catboost import CatBoostClassifier

# oof_cb = np.zeros(len(X))
# test_cb = np.zeros(len(test))

# for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
#     print(f'--- CatBoost Fold {fold}/{N_FOLDS} ---')
#     X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
#     X_te = test_n.drop(columns=[TARGET])

#     te = TargetEncoder(cols_to_encode=INTER, cv=5, smooth='auto',
#                        aggs=['mean'], drop_original=True)
#     X_tr = te.fit_transform(X_tr, y_tr)
#     X_val = te.transform(X_val)
#     X_te  = te.transform(X_te)

#     model = CatBoostClassifier(
#         iterations=10000, learning_rate=0.01, depth=5,
#         colsample_bylevel=0.5, subsample=0.8, reg_lambda=0.001,
#         early_stopping_rounds=200, eval_metric='AUC',
#         random_seed=42, thread_count=-1, verbose=1000
#     )
#     model.fit(
#         X_tr, y_tr,
#         eval_set=(X_val, y_val),
#         cat_features=CATS,  # already factorised ints
#         use_best_model=True
#     )

#     val_pred = model.predict_proba(X_val)[:, 1]
#     oof_cb[val_idx] = val_pred
#     test_cb += model.predict_proba(X_te)[:, 1] / N_FOLDS
#     print(f'CatBoost Fold {fold} AUC: {roc_auc_score(y_val, val_pred):.4f}')

# overall_cb = roc_auc_score(y, oof_cb)
# print(f'====================\nCatBoost OOF AUC: {overall_cb:.4f}\n====================')
# pd.DataFrame({'id': train.id, TARGET: oof_cb}).to_csv(f'oof_catboost_cv_{overall_cb:.4f}.csv', index=False)
# pd.DataFrame({'id': test.id, TARGET: test_cb}).to_csv(f'test_catboost_cv_{overall_cb:.4f}.csv', index=False)

In [28]:
# # notebook cell
# !pip install --force-reinstall "tensorflow==2.15.1" "keras<2.16" "pytorch-tabnet==4.1.0"

In [29]:
# pip install --force-reinstall "tensorflow==2.15.0" "keras<2.16"

In [30]:
# # =====  TabNet  (4.1.0 TF-API)  =====
# from tabnet import TabNetClassifier

# oof_tab = np.zeros(len(X))
# test_tab = np.zeros(len(test))

# # build TF feature columns once
# feat_cols = []
# for c in X.columns:
#     if c in CATS:
#         feat_cols.append(
#             tf.feature_column.categorical_column_with_identity(
#                 key=c, num_buckets=int(X[c].max() + 1)
#             )
#         )
#     else:
#         feat_cols.append(tf.feature_column.numeric_column(key=c))

# for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
#     print(f'--- TabNet Fold {fold}/{N_FOLDS} ---')
#     X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
#     X_te = test_n.drop(columns=[TARGET])

#     te = TargetEncoder(cols_to_encode=INTER, cv=5, smooth='auto',
#                        aggs=['mean'], drop_original=True)
#     X_tr = te.fit_transform(X_tr, y_tr)
#     X_val = te.transform(X_val)
#     X_te  = te.transform(X_te)

#     clf = TabNetClassifier(
#         feature_columns=feat_cols,
#         num_classes=1,               # binary
#         feature_dim=64,
#         output_dim=32,
#         num_decision_steps=3,
#         relaxation_factor=1.3,
#         batch_momentum=0.98,
#         virtual_batch_size=512,
#         sparsity_coefficient=1e-5
#     )
#     clf.fit(
#         X_tr.astype(np.float32), y_tr.astype(np.float32),
#         eval_set=[(X_val.astype(np.float32), y_val.astype(np.float32))],
#         eval_metric='auc', max_epochs=1000, patience=50,
#         batch_size=4096
#     )

#     val_pred = clf.predict(X_val.astype(np.float32)).squeeze()
#     oof_tab[val_idx] = val_pred
#     test_tab += clf.predict(X_te.astype(np.float32)).squeeze() / N_FOLDS
#     print(f'TabNet Fold {fold} AUC: {roc_auc_score(y_val, val_pred):.4f}')

# overall_tab = roc_auc_score(y, oof_tab)
# print(f'====================\nTabNet OOF AUC: {overall_tab:.4f}\n====================')
# pd.DataFrame({'id': train_org.id, TARGET: oof_tab}).to_csv(f'oof_tabnet_cv_{overall_tab:.4f}.csv', index=False)
# pd.DataFrame({'id': test.id, TARGET: test_tab}).to_csv(f'test_tabnet_cv_{overall_tab:.4f}.csv', index=False)

In [31]:
# pip install "tensorflow<2.16" -U